# FOF99 分析脚本 Runner

这个 notebook 只负责调用 `src/fof99/analyze_net_values.py`，不复制分析逻辑。脚本更新后，这里会自动使用最新逻辑。

推荐先运行 benchmark 名单，再按需要生成产品指标/HTML。

In [ ]:
from pathlib import Path
import csv
import subprocess
import sys


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "fof99" / "analyze_net_values.py").exists():
            return candidate
    fallback = Path(r"E:\策略\web-auto")
    if (fallback / "src" / "fof99" / "analyze_net_values.py").exists():
        return fallback
    raise FileNotFoundError("找不到 analyze_net_values.py，请确认 notebook 在 web-auto 项目内运行。")


ROOT = find_project_root()
SCRIPT = ROOT / "src" / "fof99" / "analyze_net_values.py"
OUTPUT_DIR = ROOT / "output" / "fof99-analysis"

print("Python:", sys.executable)
print("Project:", ROOT)
print("Script:", SCRIPT)

In [ ]:
def run_analyze(*args: str, timeout: int | None = None) -> int:
    """Run analyze_net_values.py with this notebook kernel's Python."""
    cmd = [sys.executable, str(SCRIPT), *map(str, args)]
    print("$", " ".join(cmd))
    process = subprocess.Popen(
        cmd,
        cwd=str(ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
        return process.wait(timeout=timeout)
    except KeyboardInterrupt:
        process.terminate()
        raise


def preview_csv(path: Path, n: int = 5) -> None:
    if not path.exists():
        print("文件不存在:", path)
        return
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        for index, row in enumerate(reader):
            print(row)
            if index >= n:
                break

## 1. 只查看 Benchmark 名单

这个命令不计算净值指标，只扫元数据并列出 benchmark 名称。

In [ ]:
run_analyze("--list-benchmarks")

## 2. 生成分析结果

参数说明：

- `SOURCE`: `auto` 会优先读取单产品 JSON；`csv` 只读当前汇总 CSV。
- `MIN_PRODUCT_SAMPLES`: 进入报表的最少净值样本数。全量 JSON 很大，建议先用 20 或 50。
- `SKIP_BENCHMARKS`: `True` 时只生成产品指标，不生成 benchmark。
- `JSON_WORKERS`: 默认 1。之前实测这台机器并发读 JSON 更慢，先保持 1。

In [ ]:
SOURCE = "auto"
MIN_PRODUCT_SAMPLES = 20
SKIP_BENCHMARKS = False
MIN_BENCHMARK_PRODUCTS = 2
JSON_WORKERS = 1

args = [
    "--source", SOURCE,
    "--min-product-samples", str(MIN_PRODUCT_SAMPLES),
    "--min-benchmark-products", str(MIN_BENCHMARK_PRODUCTS),
    "--json-workers", str(JSON_WORKERS),
]
if SKIP_BENCHMARKS:
    args.append("--skip-benchmarks")

exit_code = run_analyze(*args)
print("\nexit_code =", exit_code)

## 3. 快速预览输出

In [ ]:
products_csv = OUTPUT_DIR / "products.csv"
benchmarks_csv = OUTPUT_DIR / "benchmarks.csv"
html_path = OUTPUT_DIR / "index.html"

print("HTML:", html_path)
print("products.csv:", products_csv)
print("benchmarks.csv:", benchmarks_csv)

print("\nproducts.csv preview:")
preview_csv(products_csv, n=5)

print("\nbenchmarks.csv preview:")
preview_csv(benchmarks_csv, n=5)

## 4. 可选：用 pandas 查看

如果当前 kernel 安装了 pandas，可以用下面这格直接查看表格；没装也不影响上面的流程。

In [ ]:
try:
    import pandas as pd
except ImportError:
    print("当前 Python 环境没有 pandas。")
else:
    products = pd.read_csv(products_csv)
    display(products.head())